In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import spikeinterface as si
import matplotlib.pyplot as plt
import os
from matplotlib.backends.backend_pdf import PdfPages

from tqdm import tqdm

In [1]:
import sys
import spikeinterface as si
import matplotlib.pyplot as plt
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import spikeinterface.sorters as ss
import spikeinterface.widgets as sw
import spikeinterface.qualitymetrics as sqm
import json
import probeinterface

from probeinterface import Probe, ProbeGroup
from probeinterface.plotting import plot_probe, plot_probegroup
from probeinterface import generate_dummy_probe, generate_linear_probe
from probeinterface import write_probeinterface, read_probeinterface
from probeinterface import write_prb, read_prb

In [2]:
recording_raw = se.read_blackrock(file_path='/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_062422_natural_image_001.ns4')
recording_recorded = recording_raw.remove_channels(['31', '32'])


In [4]:
probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
recording_recorded = recording_recorded.set_probegroup(probe_30channel)
recording_cmr = recording_recorded
recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
recording_preprocessed = recording_cmr.save(format="binary")

Use cache_folder=/tmp/spikeinterface_cache/tmp_wa_po5k/8ES4TYMP
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=585.94 KiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

In [5]:
output_folder = f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/062422"
os.makedirs(f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/062422", exist_ok=True)
sorting_kilosort4 = ss.run_sorter(sorter_name="kilosort4", recording=recording_preprocessed, folder=output_folder + "/kilosort4")
analyzer_kilosort4 = si.create_sorting_analyzer(sorting=sorting_kilosort4, recording=recording_preprocessed, format='binary_folder', folder=f'{output_folder}/analyzer_kilosort4_binary')

extensions_to_compute = [
        "random_spikes",
        "waveforms",
        "noise_levels",
        "templates",
        "spike_amplitudes",
        "unit_locations",
        "spike_locations",
        "correlograms",
        "template_similarity"
    ]

extension_params = {
    "unit_locations": {"method": "center_of_mass"},
    "spike_locations": {"ms_before": 0.1},
    "correlograms": {"bin_ms": 0.1},
    "template_similarity": {"method": "cosine_similarity"}
}

analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

qm_params = sqm.get_default_qm_params()
analyzer_kilosort4.compute("quality_metrics", qm_params)

import spikeinterface.exporters as sexp
sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)
    

100%|██████████| 5/5 [01:31<00:00, 18.34s/it]


estimate_sparsity (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

compute_waveforms (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

Compute : spike_amplitudes + spike_locations (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

...

write_binary_recording (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/63 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/63 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/2601 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/062422/phy_folder_for_kilosort/params.py


In [6]:
for date in ['052422', '072322', '082322', '092422', '102122', '112022', '122022']:
    recording_raw = se.read_blackrock(file_path=f'/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_{date}_natural_image_001.ns4')
    recording_recorded = recording_raw.remove_channels(['31', '32'])
    probe_30channel = read_probeinterface('/media/ubuntu/sda/data/probe.json')
    recording_recorded = recording_recorded.set_probegroup(probe_30channel)
    recording_cmr = recording_recorded
    recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
    recording_cmr = spre.common_reference(recording_f, reference="global", operator="median")
    recording_preprocessed = recording_cmr.save(format="binary")
    output_folder = f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/{date}"
    os.makedirs(f"/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/{date}", exist_ok=True)
    sorting_kilosort4 = ss.run_sorter(sorter_name="kilosort4", recording=recording_preprocessed, folder=output_folder + "/kilosort4")
    analyzer_kilosort4 = si.create_sorting_analyzer(sorting=sorting_kilosort4, recording=recording_preprocessed, format='binary_folder', folder=f'{output_folder}/analyzer_kilosort4_binary')

    extensions_to_compute = [
            "random_spikes",
            "waveforms",
            "noise_levels",
            "templates",
            "spike_amplitudes",
            "unit_locations",
            "spike_locations",
            "correlograms",
            "template_similarity"
        ]

    extension_params = {
        "unit_locations": {"method": "center_of_mass"},
        "spike_locations": {"ms_before": 0.1},
        "correlograms": {"bin_ms": 0.1},
        "template_similarity": {"method": "cosine_similarity"}
    }

    analyzer_kilosort4.compute(extensions_to_compute, extension_params=extension_params)

    qm_params = sqm.get_default_qm_params()
    analyzer_kilosort4.compute("quality_metrics", qm_params)

    import spikeinterface.exporters as sexp
    sexp.export_to_phy(analyzer_kilosort4, output_folder + "/phy_folder_for_kilosort", verbose=True)

Use cache_folder=/tmp/spikeinterface_cache/tmpdhm4e5a0/VLNXYUMR
write_binary_recording 
engine=process - n_jobs=1 - samples_per_chunk=10,000 - chunk_memory=585.94 KiB - total_memory=585.94 KiB - chunk_duration=1.00s


write_binary_recording (no parallelization):   0%|          | 0/2480 [00:00<?, ?it/s]

100%|██████████| 5/5 [04:54<00:00, 58.91s/it]


estimate_sparsity (no parallelization):   0%|          | 0/2480 [00:00<?, ?it/s]

compute_waveforms (no parallelization):   0%|          | 0/2480 [00:00<?, ?it/s]

...

write_binary_recording (no parallelization):   0%|          | 0/2540 [00:00<?, ?it/s]

Fitting PCA:   0%|          | 0/72 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/72 [00:00<?, ?it/s]

extract PCs (no parallelization):   0%|          | 0/2540 [00:00<?, ?it/s]

Run:
phy template-gui  /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/122022/phy_folder_for_kilosort/params.py
